# Healthcare LoRA SFT — live demo

One LoRA adapter on `Qwen2.5-1.5B-Instruct` doing two clinical tasks:

1. **Classification** — symptom description → one of 22 conditions
2. **Summarization** — doctor–patient dialogue → structured clinical note

Runs entirely on this Mac. No API calls, no data leaves the machine.

The adapter is **70 MB against a 3 GB base model** — 1.18% of the parameters.
It is loaded once below and then toggled on and off in place, so every
comparison you see uses the *same weights in the same process*, differing only
by whether the adapter bypass is active.

> Decision-support demonstration only. Not a medical device, not medical advice.

**Before running:** select the **`healthcare-sft (.venv)`** kernel. Any other
kernel will use a different Python environment.

## Setup

In [ ]:
import json, os, sys, textwrap, torch
from transformers.utils import logging as hf_logging

hf_logging.set_verbosity_error()  # keep the demo output clean
from pathlib import Path

# Jupyter sets the working directory to the notebook's folder, but this project
# uses repo-root-relative paths ("data/...", "src.prompts"). Step up if needed.
if not Path("src").exists():
    os.chdir("..")
sys.path.insert(0, str(Path.cwd()))
print("working directory:", Path.cwd())
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

from src.prompts import build_classification_messages, build_summarization_messages
from src.tokenization import render_prompt

BASE_MODEL = "Qwen/Qwen2.5-1.5B-Instruct"
ADAPTER    = "adapters/qwen-healthcare-lora"
LABELS     = json.loads(Path("data/labels.json").read_text())

print(f"{len(LABELS)} conditions:")
print(textwrap.fill(", ".join(LABELS), 88))

### Load the model — run once, takes ~60 seconds

Deliberately **no** `device_map="auto"`: on an 8 GB Mac accelerate offloads
layers to disk as placeholder (`meta`) parameters, and PEFT then loads the
adapter into placeholders — a silent no-op that discards the adapter entirely.
We load on CPU where every weight is real, attach the adapter, then move to the
GPU.

In [ ]:
device = "mps" if torch.backends.mps.is_available() else "cpu"
dtype  = torch.float16 if device == "mps" else torch.float32

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
model = AutoModelForCausalLM.from_pretrained(BASE_MODEL, dtype=dtype)
model = PeftModel.from_pretrained(model, ADAPTER)

meta = [n for n, p in model.named_parameters() if p.device.type == "meta"]
assert not meta, f"{len(meta)} params on meta device — adapter would be ignored"

model.to(device).eval()

trainable = sum(p.numel() for n, p in model.named_parameters() if "lora_" in n)
total     = sum(p.numel() for p in model.parameters())
print(f"loaded on {device} ({dtype})")
print(f"LoRA parameters: {trainable:,} / {total:,} = {100*trainable/total:.2f}%")

### Helper

`tuned=False` wraps generation in `model.disable_adapter()`, which switches the
LoRA bypass off. Same model object, same memory — only the adapter changes.

In [ ]:
@torch.no_grad()
def ask(task, text, tuned=True, max_new_tokens=None):
    if task == "classify":
        messages = build_classification_messages(text, LABELS)
        max_new_tokens = max_new_tokens or 16
    else:
        messages = build_summarization_messages(text)
        max_new_tokens = max_new_tokens or 256

    inputs = tokenizer(render_prompt(messages, tokenizer), return_tensors="pt").to(model.device)

    def _gen():
        out = model.generate(**inputs, max_new_tokens=max_new_tokens,
                             do_sample=False, pad_token_id=tokenizer.pad_token_id)
        return tokenizer.decode(out[0, inputs["input_ids"].shape[1]:],
                                skip_special_tokens=True).strip()

    if tuned:
        return _gen()
    with model.disable_adapter():
        return _gen()


def compare(task, text, width=88):
    print("INPUT")
    print(textwrap.fill(text, width)); print()
    for label, tuned in [("BASE MODEL (adapter off)", False), ("TUNED (adapter on)", True)]:
        print("=" * width); print(label); print("=" * width)
        print(ask(task, text, tuned=tuned)); print()

---
# Demo 1 — Classification

Symptom description in, one of 22 condition labels out.

In [ ]:
compare("classify",
    "I have a burning feeling when I urinate and I need to go constantly.")

**What to notice.** The base model is not ignorant — it usually knows the
condition. What it lacks is the discipline to answer with exactly one label from
the given vocabulary, in the expected language and format. That gap is what SFT
closed.

On the held-out test set: accuracy **30.7% → 92.0%**, and outputs that were not
a valid label at all **15.1% → 0%**.

In [ ]:
for symptoms in [
    "I've had a high fever with chills for four days, severe headache, and pain behind my eyes.",
    "There are itchy silvery scaly patches on my elbows and knees that keep coming back.",
    "My skin turned yellow, my urine is very dark, and I feel exhausted and nauseous.",
]:
    print(textwrap.fill(symptoms, 88))
    print(f"  base  : {ask('classify', symptoms, tuned=False)}")
    print(f"  tuned : {ask('classify', symptoms, tuned=True)}")
    print()

**Be honest about what you just saw.** On obvious presentations the base model
often gets the condition right — it knows the medicine. The dengue case is the
interesting one: the base model said *pneumonia*, the tuned model said *dengue*.

The reliable gains are (a) format — always one label, always in English, always
from the list — and (b) consistency across all 212 held-out examples rather than
three cherry-picked ones. That is why the numbers below matter more than any
single example.

---
# Demo 2 — Summarization

The *same adapter*, selected by a different task tag in the prompt. Dialogue in,
four-section clinical note out.

In [ ]:
DIALOGUE = """Doctor: What brings you in today?
Patient: My right knee has been aching for about three weeks now, worse when I go up stairs.
Doctor: Any injury that you recall?
Patient: No, it just started on its own. I'm 58 and I've had some arthritis in my hands.
Doctor: Any swelling or redness?
Patient: A little puffy in the mornings.
Doctor: Let's get an X-ray and start you on anti-inflammatories, and I'd like you to see physiotherapy."""

compare("summarize", DIALOGUE)

**What to notice.** `Diagnosis: N/A` — the doctor never gave one, and the
model did not invent one. The system prompt says to write N/A for anything the
conversation does not cover, and it obeyed rather than hallucinating. In a
clinical note that restraint matters more than fluency.

On the held-out set: ROUGE-1 **0.480 → 0.715**, section adherence
**96% → 100%**.

---
# Demo 3 — Try your own

Edit the text and re-run.

In [ ]:
print(ask("classify", "I get short of breath and wheeze badly whenever I exercise or it gets cold."))

In [ ]:
print(ask("summarize", """Doctor: How have the headaches been?
Patient: Still getting them most mornings, right behind my left eye.
Doctor: Any nausea or light sensitivity?
Patient: Yes, bright light makes it much worse.
Doctor: That sounds like migraine. Let's try sumatriptan and keep a headache diary."""))

---
# Results on held-out data

Both models evaluated on identical prompts, identical greedy decoding, identical
4-bit quantization — differing only in whether the adapter was attached.

In [ ]:
m = json.loads(Path("results/metrics.json").read_text())

def table(sub, keys, pct):
    print(f"{'metric':<22}{'base':>10}{'tuned':>10}{'delta':>12}")
    print("-" * 54)
    for k, name in keys:
        b, t = m["base"][sub][k], m["tuned"][sub][k]
        f = (lambda x: f"{x:.1%}") if pct.get(k, True) else (lambda x: f"{x:.3f}")
        print(f"{name:<22}{f(b):>10}{f(t):>10}{f(t-b):>12}")
    print()

print(f"CLASSIFICATION  ({m['base']['classification']['n']} held-out examples)")
table("classification", [("accuracy","accuracy"),("macro_f1","macro-F1"),
                         ("invalid_rate","invalid-label rate")], {})
print(f"SUMMARIZATION   ({m['base']['summarization']['n']} held-out dialogues)")
table("summarization", [("rouge1","ROUGE-1"),("rouge2","ROUGE-2"),("rougeL","ROUGE-L"),
                        ("section_adherence","section adherence")],
      {"rouge1": False, "rouge2": False, "rougeL": False})

### Confusion matrices

22 conditions plus an explicit **INVALID** column. That column is load-bearing:
scikit-learn silently drops any prediction outside the label list, so without it
most of the base model's examples would vanish from the plot and the matrix
would look fine while describing a fraction of the data.

In [ ]:
from IPython.display import Image, display
for tag in ["base", "tuned"]:
    print(tag.upper())
    display(Image(f"results/confusion_matrix_{tag}.png", width=780))

---
# What the adapter actually is

In [ ]:
cfg = json.loads(Path(f"{ADAPTER}/adapter_config.json").read_text())
for k in ["base_model_name_or_path", "r", "lora_alpha", "lora_dropout", "target_modules"]:
    print(f"{k:26} {cfg[k]}")

size = Path(f"{ADAPTER}/adapter_model.safetensors").stat().st_size
print(f"\nadapter_model.safetensors  {size/1e6:.0f} MB   (base model is ~3 GB)")

The adapter is a **diff, not a model** — `adapter_config.json` names the exact
base model it patches, and it is meaningless without it.

That is also what makes the architecture interesting commercially: one 3 GB base
model held in memory, with a folder of 70 MB adapters swapped per request — one
per client, specialty, or note format. Twenty behaviours cost 3 GB + 20 × 70 MB,
not 20 × 3 GB.

Full method, results and limitations: [`README.md`](../README.md).
Concept-by-concept explanation: [`docs/CONCEPTS.md`](../docs/CONCEPTS.md).